In [1]:
## Libraries, modules, black magic
import os
import soundfile as sf
import numpy as np
from scipy.signal import square, ShortTimeFFT
from scipy.signal.windows import hann

## Parameters
directory = 'f_tmp_segments/' # Look for recording chunks in this directory
chunk_offset = 1500 - 300 # In seconds, this is the incremental offset for each chunk (calculated as duration - overlap)

## Functions
# Spectrogram bandpass filter
def spec_bandpass(spec, f_max, f_bins, f_low, f_high):
    f_bl = int(f_low/f_max*f_bins) # Low frequency bin
    f_bh = int(f_high/f_max*f_bins) # High frequency bin
    return spec[f_bl:f_bh,:]

# Fill gaps in a sequence of signals
def fill_gaps(arr, n):
    result = arr.copy()
    i = 0
    while i < len(result):
        if result[i] == 0: # Found the start of a sequence of 0s            
            start = i
            while i < len(result) and result[i] == 0: # Find the end of this sequence
                i += 1
            end = i            
            if end - start < n: # If the length of the sequence is shorter than n, replace with 1s
                for j in range(start, end):
                    result[j] = 1
        else:
            i += 1    
    return result

# Detect sequences from a "detection array"
def detect_sequences(bin_list):
    sequences = []
    start_idx = None    
    for i, num in enumerate(bin_list): # Go through detection array
        if num == 1:
            if start_idx is None:  # Beginning of a new sequence of 1s
                start_idx = i
        elif num == 0 and start_idx is not None:  # End the sequence when we hit a 0 after detecting 1s
            sequences.append((start_idx, i - 1))  # End of sequence
            start_idx = None    
    if start_idx is not None:     # If we end with a sequence of 1s and it's still open, close it
        sequences.append((start_idx, len(bin_list) - 1))    
    return sequences

# Filter out sequences where the difference between end and start is less than min_len
def delete_short_sequences(arr, min_dur):
    tmp_list = []
    for row in arr:
        if np.all(np.abs(np.diff(row)) > min_dur):
            tmp_list.append(row)            
    return np.array(tmp_list)

# Filter out sequences that are at the beginning (start_idx == 0) or at the end (end_idx == length - 1)
def delete_edge_sequences(sequences):
    length = len(sig_detected)
    return [seq for seq in sequences if seq[0] != 0 and seq[1] != length - 1]

# Delete duplicate sequences
def delete_duplicates(arr, min_diff):
    result = [arr[0]]  # Always keep the first row
    for i in range(1, len(arr)):
        prev_row = arr[i - 1]
        curr_row = arr[i]        
        # Check if the difference between the columns of the current and previous row is less than or equal to n
        if abs(curr_row[0] - prev_row[0]) > min_diff or abs(curr_row[1] - prev_row[1]) > min_diff:
            result.append(curr_row)  # Keep the row if the condition is met
    return np.array(result) # Convert the list back to a numpy array

In [2]:
## Loop through each chunk
sequences_all = np.empty((0, 2)) # Array where all sequences will be stored
for filename in os.listdir(directory):
    file_path = os.path.join(directory, filename)
    if os.path.isfile(file_path):  # check if it's a file (not a directory)    
        
        ## Determine chunk number
        chunk_no = int(filename.split("_")[0]) # Split the filename by "_", the number is the first part before "_segment"

        ## Read chunk
        data, fs = sf.read(directory + filename) # fs is sample rate, should be 48 kHz mono

        ## Generate Spectrogram (2D FFT)
        win_size = 4096 # 4096 seems to work better than 2048
        win = hann(win_size) # Hann window
        SFT = ShortTimeFFT(win, hop=win_size//2, fs=fs) # 50% overlap with hop=win_size/2
        Sx2 = SFT.spectrogram(data)  # Spectrogram (absolute square of STFT)
        Sx2[Sx2 == 0] = np.min(Sx2[np.nonzero(Sx2)]) # Substitute zeros with minimum non-zero value
        Sx2_dB = 10*np.log10(Sx2) # Spectrogram in dB
        t_min, t_max, f_min, f_max = SFT.extent(len(data), center_bins=True) # Frequency and time boundaries
        f_bins, t_bins = Sx2.shape # Number of frequency and time bins

        ## Measure background noise level
        noise = spec_bandpass(Sx2_dB, f_max, f_bins, 6300, 6700)
        noise_mean = np.mean(noise) # Mean of the whole noise 2D spectrogram
        noise_std = np.std(noise) # Standard dev. of the whole noise 2D spectrogram

        ## Measure signal level at low frequencies
        sig_low = spec_bandpass(Sx2_dB, f_max, f_bins, 7300, 7700)
        sig_level_l = np.max(sig_low, axis=0) # Maximum level for each time sample

        ## Measure signal level at high frequencies
        sig_high = spec_bandpass(Sx2_dB, f_max, f_bins, 10000, 10600)
        sig_level_h = np.max(sig_high, axis=0) # Maximum level for each time sample

        ## Create "detection" array
        threshold = noise_mean + noise_std*3 # Threshold level to detect a signal
        above_threshold = ((sig_level_l > threshold) | (sig_level_h > threshold)).astype(int) # Array of detections (1) and non-detections (0)
        max_zeros = int(fs/win_size*2*5) # 5 seconds, very conservative, I found that 1 second is sufficient 
        sig_detected = fill_gaps(above_threshold, max_zeros)

        ## Detect signal sequences
        tmp_sequences = detect_sequences(sig_detected)

        ## Filter out sequences at the beginning and end of recording (they are truncated and thus incomplete)
        tmp_sequences = delete_edge_sequences(tmp_sequences)

        ## Turn sequences expressed as sample numbers into time in seconds, make corrections
        sequence_timing = np.array(tmp_sequences)/(fs/(win_size/2)) # Conversion to seconds
        sequence_timing = sequence_timing + chunk_no*1200 # Add shift due to segmentation
        sequence_timing[:, 0] = sequence_timing[:, 0] - 10 # Add 10 s of buffer before each segment for safety
        sequence_timing[:, 1] = sequence_timing[:, 1] + 10 # Add 10 s of buffer after each segment for safety
        sequence_timing = np.round(sequence_timing, 3) # Round to 3rd decimal digit (1 ms), the mimimum time step at 48kHz and 512 FFT samples is 0.010666...

        ## Add sequences to the full array
        sequences_all = np.concatenate((sequences_all, sequence_timing))
        print("Processed segment " + str(chunk_no), end="\r")
print("Done!")

Done!ssed segment 61


In [3]:
## Filter out noise (typically, this is due to in-game sounds)
min_duration = 28.56*0.9 + 20 # In seconds, minimum duration of a sequence (-10% margin) + 2x10 s buffers
sequences_all = delete_short_sequences(sequences_all, min_duration) 

## Sort by start sample
sequences_all = sequences_all[sequences_all[:, 0].argsort()]

## Delete duplicate detections
min_time_diff = 26.6 # In seconds, this is the minimum time differences between two sequences start or end times
sequences_all = delete_duplicates(sequences_all, min_time_diff)

## Export sequence timings to csv
out_filename = "F_sequences_timing.csv"
np.savetxt(out_filename, sequences_all, delimiter=',', fmt='%.3f')

print("Done processing all sequences, results in " + out_filename)

Done processing all sequences, results in F_sequences_timing.csv
